# 04 — Counterfactual Pricing & Promotion Simulator

**目标**：把 03 frozen 的弹性变成可执行的 price/promo 建议。**对每个 eligible brand-size-store cell**：在价格 × 促销网格上预测 Q、revenue、profit，输出 top-10 行动建议 + sensitivity band。

**主模型系数**（来自 `reports/demand_model_summary.md` FROZEN block）：
- β_own = -1.73,  β_cross = +0.65,  promo uplift = +0.43,  smearing S = 1.14

**反事实预测公式**（cell-anchored，绕过 FE）：
$$ \hat{Q}(p, m \mid \text{cell}) = \bar{Q}_\text{cell} \cdot \exp\Big[\beta_\text{own}\bigl(\log p - \log \bar{p}_\text{cell}\bigr) + \theta\,(m - \bar{m}_\text{cell})\Big] $$

**为什么不再额外乘 S**：retransformation bias $E[\exp(\varepsilon)] = S$ 是把 FE 模型的 log 预测搬回 level 时需要的修正。这里我们 anchor 在观察到的 $\bar{Q}_\text{cell}$ —— 它已经是 level（$\bar{Q} = E[\exp(\alpha + \gamma + X\beta + \varepsilon)]$），smearing 已经被吸收。再乘 S 会双重计数。S 仍记录在 frozen block 里，供 FE-based 预测（如 03 的 holdout 路径）使用。

`p_comp` 默认 hold at 历史均值（cross 项进入 sensitivity）。

**Eligibility filter**（在选 top-N 之前先剔除脏 cell）：
1. `brand_confidence ∈ {high, medium}` —— 已在 03 数据集层面应用
2. `size_kind ∈ {oz, oz_bundle}` —— 已在 03 数据集层面应用
3. 历史观察 ≥ 52 周
4. `unit_cost > 0`
5. 当前 `unit_price` 落在 brand-size 的 IQR 内（剔除清仓尾巴）
6. cell 内 `unit_price_cv` 不超过 0.5（剔除高离散）
7. 平均 `quantity ≥ 5`

**产出**：
1. `data/processed/cell_baselines.parquet` — 进入优化器的 cell 级输入
2. `data/processed/top_recommendations.csv` — top-10 价格行动 + 预测 lift
3. `data/processed/sensitivity_grid.csv` — β_own × β_cross × promo 三维 sensitivity
4. `reports/counterfactual_summary.md` — 摘要 + 可信度边界
5. `reports/figures/counterfactual_*.png` — top-10 lift 图、sensitivity heatmap


In [ ]:
from __future__ import annotations
from pathlib import Path
from datetime import datetime
import sys, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED = PROJECT_ROOT / 'data' / 'processed'
REPORTS   = PROJECT_ROOT / 'reports'
FIGURES   = REPORTS / 'figures'
for p in (PROCESSED, FIGURES): p.mkdir(parents=True, exist_ok=True)

summary_lines: list[str] = []
def log(msg: str) -> None:
    print(msg); summary_lines.append(msg)

log('# Counterfactual Simulator Summary')
log(f'Generated: {datetime.now().isoformat(timespec="seconds")}')
log('')


## 1. Load coefficients + modeling dataset


In [ ]:
coef_df = pd.read_csv(PROCESSED / 'model_coefficients.csv')
mdl_data = pd.read_parquet(PROCESSED / 'demand_modeling_dataset.parquet')

main = coef_df.set_index('model').loc['baseline_with_cross']
BETA_OWN   = float(main['beta_own_price'])
BETA_CROSS = float(main['beta_cross_price'])
THETA      = float(main['beta_promo'])
SMEARING   = float(main['smearing'])

log('## 1. Load')
log(f'- main model = baseline_with_cross')
log(f'- β_own = {BETA_OWN:.4f}, β_cross = {BETA_CROSS:.4f}, '
    f'θ_promo = {THETA:.4f}, S = {SMEARING:.4f}')
log(f'- modeling dataset rows: {len(mdl_data):,}')


## 2. Build cell-level baselines + apply eligibility filter

每个 `brand_final × size_oz_rounded × STORE` cell 的 baseline：
- `n_weeks`：观察周数
- `mean_q`、`mean_p`、`mean_cost`、`mean_promo`、`mean_log_comp_price`
- `p_min`、`p_max`、`unit_price_cv`：定价边界
- `revenue_total`：用于排序候选 cell


In [ ]:
# Aggregate to cell level (brand, size, store)
g = mdl_data.groupby(['brand_final', 'size_oz_rounded', 'STORE'], observed=True)
cell = g.agg(
    n_weeks            = ('WEEK', 'nunique'),
    mean_q             = ('quantity', 'mean'),
    mean_p             = ('unit_price', 'mean'),
    p_min              = ('unit_price', 'min'),
    p_max              = ('unit_price', 'max'),
    unit_price_std     = ('unit_price', 'std'),
    mean_cost          = ('unit_cost', 'mean'),
    mean_promo         = ('promo_any_int', 'mean'),
    mean_log_comp_price= ('log_comp_price', 'mean'),
    revenue_total      = ('quantity', lambda q: float((q * mdl_data.loc[q.index, 'unit_price']).sum())),
).reset_index()
cell['unit_price_cv'] = cell['unit_price_std'] / cell['mean_p']

log('## 2. Cell baselines + eligibility')
log(f'- raw cells (brand-size-store): {len(cell):,}')

# ---- Eligibility filter ----
# (1)+(2) brand_confidence/size_kind already applied upstream
# (3) >= 52 weeks
mask3 = cell['n_weeks'] >= 52
# (4) unit_cost > 0
mask4 = cell['mean_cost'] > 0
# (5) current mean_p within brand-size IQR
bs_q = cell.groupby(['brand_final', 'size_oz_rounded'], observed=True)['mean_p'].transform(
    lambda s: pd.Series([s.quantile(0.25), s.quantile(0.75)] * (len(s)//2 + 1)).iloc[:len(s)].values
)
# Compute Q1/Q3 explicitly to avoid hacky transform
q1 = cell.groupby(['brand_final', 'size_oz_rounded'], observed=True)['mean_p'].transform('quantile', 0.25)
q3 = cell.groupby(['brand_final', 'size_oz_rounded'], observed=True)['mean_p'].transform('quantile', 0.75)
mask5 = (cell['mean_p'] >= q1) & (cell['mean_p'] <= q3 + 1e-9)
# Edge: groups with <4 cells -> always pass (q1 == q3)
mask5 = mask5 | (q1 == q3)
# (6) cv <= 0.5
mask6 = (cell['unit_price_cv'].fillna(0) <= 0.5)
# (7) mean_q >= 5
mask7 = cell['mean_q'] >= 5

reasons = pd.DataFrame({
    'fail_n_weeks_lt_52': ~mask3,
    'fail_cost_le_0'    : ~mask4,
    'fail_outside_IQR'  : ~mask5,
    'fail_cv_gt_0.5'    : ~mask6,
    'fail_mean_q_lt_5'  : ~mask7,
})
log(f'- failure counts (any cell can fail multiple):')
for c in reasons.columns:
    log(f'    · {c}: {int(reasons[c].sum()):,}')

eligible_mask = mask3 & mask4 & mask5 & mask6 & mask7
cell_elig = cell.loc[eligible_mask].copy().reset_index(drop=True)
log(f'- eligible cells: {len(cell_elig):,} ({100*len(cell_elig)/len(cell):.1f}%)')


## 3. Define cell-anchored demand function $\hat Q(p, m \mid \text{cell})$

$$\hat Q(p, m) = \bar Q \cdot \exp\Big[\beta_\text{own}(\log p - \log \bar p) + \theta(m - \bar m)\Big]$$

`smearing=1.0` by default —— 见 §4 关于为什么 cell-anchor 不需要再乘 S。`p_comp` 在主表中保持历史均值不变（即 $\beta_\text{cross}$ 项 cancel）；进入 sensitivity 时单独打开 `log_p_comp_delta`。


In [ ]:
def predict_q(p, m, *, mean_q, mean_p, mean_promo,
              beta_own=BETA_OWN, theta=THETA, smearing=1.0,
              beta_cross=0.0, log_p_comp_delta=0.0):
    """Cell-anchored demand prediction.

    smearing defaults to 1.0 because mean_q is an empirical level mean and
    already absorbs E[exp(eps)]; multiplying by S would double-count. See §4.
    p, m, log_p_comp_delta can be scalars or arrays; mean_* are scalars per call.
    """
    p   = np.asarray(p, dtype='float64')
    m   = np.asarray(m, dtype='float64')
    log_term = (beta_own * (np.log(p) - np.log(mean_p))
                + theta    * (m - mean_promo)
                + beta_cross * log_p_comp_delta)
    return mean_q * np.exp(log_term) * smearing

# Sanity: at (p=mean_p, m=mean_promo) and default smearing=1.0 → mean_q
sanity_q = predict_q(p=cell_elig['mean_p'].iloc[0], m=cell_elig['mean_promo'].iloc[0],
                     mean_q=cell_elig['mean_q'].iloc[0],
                     mean_p=cell_elig['mean_p'].iloc[0],
                     mean_promo=cell_elig['mean_promo'].iloc[0])
log('## 3. Demand function')
log(f'- sanity (anchor → anchor): predicted Q = {float(sanity_q):.3f} '
    f'vs mean_q = {cell_elig["mean_q"].iloc[0]:.3f}  (no S multiplier)')


## 4. Smearing & calibration diagnostic

**Anchor vs FE prediction**：03 的 holdout 路径用 FE 系数 + α̂ + γ̂ 重建 log_q，必须再乘 S=1.14 才能把 log-level 推回 level。这里我们直接 anchor 在 $\bar Q_\text{cell}$（已经是 level，empirically observed），所以 `predict_q` 默认 `smearing=1.0`。

下面对比两种做法在 in-sample 上的 calibration：cell-anchor 应当 ≈ 1.0；额外乘 S 会出现 ~`S` 倍 inflation。


In [ ]:
# Predict observed-period Q for every modeling row using cell anchor
mdl_w_cell = mdl_data.merge(
    cell_elig[['brand_final','size_oz_rounded','STORE','mean_q','mean_p','mean_promo']],
    on=['brand_final','size_oz_rounded','STORE'], how='inner'
)
predicted_anchor = predict_q(
    p=mdl_w_cell['unit_price'].to_numpy(),
    m=mdl_w_cell['promo_any_int'].to_numpy(),
    mean_q=mdl_w_cell['mean_q'].to_numpy(),
    mean_p=mdl_w_cell['mean_p'].to_numpy(),
    mean_promo=mdl_w_cell['mean_promo'].to_numpy(),
    smearing=1.0,
)
predicted_with_s = predicted_anchor * SMEARING
observed = mdl_w_cell['quantity'].to_numpy()
log('## 4. Calibration')
log(f'- observed Σ Q                 = {observed.sum():,.0f}')
log(f'- cell-anchor predicted (S=1)  = {predicted_anchor.sum():,.0f}  '
    f'ratio = {predicted_anchor.sum()/observed.sum():.4f}')
log(f'- cell-anchor × S=1.14         = {predicted_with_s.sum():,.0f}  '
    f'ratio = {predicted_with_s.sum()/observed.sum():.4f}')
log('  → S=1.0 is the right choice for anchor-based prediction (no double counting).')


## 5. Compute revenue / profit at observed baseline

Profit per cell = `(p - cost) × Q`。先把每个 eligible cell 的当前 baseline profit 算出来作为对比基准。


In [ ]:
# Baseline observed economics per cell (no smearing — mean_q is empirical level)
ce = cell_elig.copy()
ce['baseline_q']      = ce['mean_q']
ce['baseline_rev']    = ce['baseline_q'] * ce['mean_p']
ce['baseline_margin'] = ce['mean_p'] - ce['mean_cost']
ce['baseline_profit'] = ce['baseline_q'] * ce['baseline_margin']

log('## 5. Baseline economics')
log(f'- mean baseline profit per cell: ${ce["baseline_profit"].mean():.2f}')
log(f'- total baseline revenue across eligible cells: ${ce["baseline_rev"].sum():,.0f}')
log(f'- total baseline profit across eligible cells: ${ce["baseline_profit"].sum():,.0f}')


## 6. Optimization constraints (price bounds, margin, promo)

- **Price grid**：`[max(p_min·0.85, cost·1.05), p_max·1.15]`，21 个网格点
- **Margin floor**：price ≥ 1.05 × cost（至少 5% gross margin）
- **Promo grid**：{0, 1}
- **Inventory scenario**：MVP 不设 hard cap；记录预测 Q 的相对涨幅，留给 memo 提醒补货


In [ ]:
GRID_N = 21
MARGIN_FLOOR_RATIO = 1.05  # price >= 1.05 * cost

def make_price_grid(p_min, p_max, cost, n=GRID_N):
    lo = max(p_min * 0.85, cost * MARGIN_FLOOR_RATIO)
    hi = p_max * 1.15
    if lo >= hi:
        return np.array([max(lo, hi)])
    return np.linspace(lo, hi, n)

# Smoke test
g0 = make_price_grid(ce['p_min'].iloc[0], ce['p_max'].iloc[0], ce['mean_cost'].iloc[0])
log('## 6. Constraints')
log(f'- price grid len (sample cell): {len(g0)}, range [{g0.min():.3f}, {g0.max():.3f}]')
log(f'- margin floor: price ≥ {MARGIN_FLOOR_RATIO} × cost')


## 7. Grid search over (price, promo) per cell


In [ ]:
def optimize_cell(row, *, beta_own=BETA_OWN, theta=THETA, smearing=1.0,
                  beta_cross=0.0, log_p_comp_delta=0.0):
    grid_p = make_price_grid(row['p_min'], row['p_max'], row['mean_cost'])
    grid_m = np.array([0, 1])
    P, M = np.meshgrid(grid_p, grid_m, indexing='ij')
    Q = predict_q(P, M,
                  mean_q=row['mean_q'], mean_p=row['mean_p'], mean_promo=row['mean_promo'],
                  beta_own=beta_own, theta=theta, smearing=smearing,
                  beta_cross=beta_cross, log_p_comp_delta=log_p_comp_delta)
    margin = P - row['mean_cost']
    profit = Q * margin
    rev    = Q * P
    flat_idx = int(np.argmax(profit))
    i, j = np.unravel_index(flat_idx, profit.shape)
    grid_hi = float(grid_p.max())
    grid_lo = float(grid_p.min())
    return {
        'opt_price':     float(P[i, j]),
        'opt_promo':     int(M[i, j]),
        'opt_q':         float(Q[i, j]),
        'opt_rev':       float(rev[i, j]),
        'opt_profit':    float(profit[i, j]),
        'opt_hits_upper':bool(P[i, j] >= grid_hi - 1e-9),
        'opt_hits_lower':bool(P[i, j] <= grid_lo + 1e-9),
    }

opt_rows = ce.apply(optimize_cell, axis=1, result_type='expand')
ce_opt = pd.concat([ce.reset_index(drop=True), opt_rows.reset_index(drop=True)], axis=1)
ce_opt['profit_lift_abs'] = ce_opt['opt_profit'] - ce_opt['baseline_profit']
ce_opt['profit_lift_pct'] = ce_opt['profit_lift_abs'] / ce_opt['baseline_profit'].clip(lower=1e-6) * 100
ce_opt['q_lift_ratio']    = ce_opt['opt_q']      / ce_opt['baseline_q'].clip(lower=1e-6)

log('## 7. Grid search')
log(f'- cells optimized: {len(ce_opt):,}')
log(f'- mean profit lift: ${ce_opt["profit_lift_abs"].mean():.2f}/cell '
    f'(median {ce_opt["profit_lift_abs"].median():.2f})')
log(f'- cells with non-trivial lift (≥5%): {int((ce_opt["profit_lift_pct"]>=5).sum()):,}')
log(f'- cells at status quo already optimal: '
    f'{int((ce_opt["profit_lift_abs"]<=1e-6).sum()):,}')
log(f'- cells whose recommendation pegs the upper price grid bound: '
    f'{int(ce_opt["opt_hits_upper"].sum()):,} '
    f'({100*ce_opt["opt_hits_upper"].mean():.1f}%) — flagged as extrapolation risk')


## 8. Top-10 recommendations

两个视图：
- **`top10_raw`** — 按 absolute profit lift 排序前 10。Memo / Streamlit 用作 "single best opportunity" 列。
- **`top10_diverse`** — 按 `brand_final × size_oz_rounded` 去重，每组只保留最高 lift 的 store，再取前 10。这是给商品组讨论时用的 portfolio 视图，避免同一 SKU 占满榜单。


In [ ]:
COLS = ['brand_final','size_oz_rounded','STORE','n_weeks',
        'mean_p','mean_promo','mean_cost','baseline_q','baseline_profit',
        'opt_price','opt_promo','opt_q','opt_profit',
        'profit_lift_abs','profit_lift_pct','q_lift_ratio','opt_hits_upper']

top10_raw = ce_opt.sort_values('profit_lift_abs', ascending=False).head(10)[COLS].round(3)
top10_diverse = (ce_opt.sort_values('profit_lift_abs', ascending=False)
                       .drop_duplicates(['brand_final','size_oz_rounded'], keep='first')
                       .head(10)[COLS].round(3))

ce_opt.round(4).to_csv(PROCESSED / 'all_recommendations.csv', index=False)
top10_raw.to_csv(PROCESSED / 'top_recommendations.csv', index=False)
top10_diverse.to_csv(PROCESSED / 'top_recommendations_diverse.csv', index=False)

log('## 8. Top-10 recommendations')
log(f'- saved: data/processed/top_recommendations.csv (raw) + top_recommendations_diverse.csv')
log('')
log('### Top-10 (raw, by profit lift)')
log('| brand | size | store | mean_p → opt_p | promo→ | Q×↑ | Δprofit | upper-bound? |')
log('|---|---|---|---|---|---|---|---|')
for _, r in top10_raw.iterrows():
    log(f'| {r["brand_final"]} | {r["size_oz_rounded"]:.2f} | {int(r["STORE"])} | '
        f'{r["mean_p"]:.2f} → {r["opt_price"]:.2f} | '
        f'{int(r["mean_promo"]>0.5)}→{int(r["opt_promo"])} | '
        f'{r["q_lift_ratio"]:.2f}× | '
        f'${r["profit_lift_abs"]:.2f} ({r["profit_lift_pct"]:.1f}%) | '
        f'{"⚠ yes" if r["opt_hits_upper"] else "no"} |')
log('')
log('### Top-10 (one per brand-size, portfolio view)')
log('| brand | size | store | mean_p → opt_p | promo→ | Q×↑ | Δprofit | upper-bound? |')
log('|---|---|---|---|---|---|---|---|')
for _, r in top10_diverse.iterrows():
    log(f'| {r["brand_final"]} | {r["size_oz_rounded"]:.2f} | {int(r["STORE"])} | '
        f'{r["mean_p"]:.2f} → {r["opt_price"]:.2f} | '
        f'{int(r["mean_promo"]>0.5)}→{int(r["opt_promo"])} | '
        f'{r["q_lift_ratio"]:.2f}× | '
        f'${r["profit_lift_abs"]:.2f} ({r["profit_lift_pct"]:.1f}%) | '
        f'{"⚠ yes" if r["opt_hits_upper"] else "no"} |')


## 9. Sensitivity analysis

对 portfolio top-10 cells 的 *recommended price* 与 *predicted profit lift* 做 β_own × β_cross × promo 网格扰动。`p_comp` 在 β_cross>0 时同步 +10% shock（"rivals raise price"），β_cross=0 时不动。


In [ ]:
beta_own_grid   = [-1.90, -1.73, -1.50]
beta_cross_grid = [0.0, 0.50, 0.65]
promo_grid      = [0.30, 0.43, 0.51]

# Pull full ce_opt rows (with p_min/p_max/cost) for the portfolio top-10
keys = top10_diverse[['brand_final','size_oz_rounded','STORE']].astype({'STORE':'int64'})
sens_target = ce_opt.merge(keys, on=['brand_final','size_oz_rounded','STORE'], how='inner')

sens_records = []
for bo in beta_own_grid:
    for bc in beta_cross_grid:
        for th in promo_grid:
            shock = 0.10 if bc > 0 else 0.0
            for _, row in sens_target.iterrows():
                res = optimize_cell(row, beta_own=bo, theta=th, beta_cross=bc,
                                    log_p_comp_delta=shock)
                sens_records.append({
                    'brand_final': row['brand_final'],
                    'size_oz_rounded': float(row['size_oz_rounded']),
                    'STORE': int(row['STORE']),
                    'beta_own': bo,
                    'beta_cross': bc,
                    'promo_theta': th,
                    'opt_price': res['opt_price'],
                    'opt_promo': res['opt_promo'],
                    'opt_profit': res['opt_profit'],
                    'baseline_profit': float(row['baseline_profit']),
                    'profit_lift_abs': res['opt_profit'] - float(row['baseline_profit']),
                })
sens = pd.DataFrame(sens_records)
sens_path = PROCESSED / 'sensitivity_grid.csv'
sens.round(4).to_csv(sens_path, index=False)

# Per-cell summary across the sensitivity grid
sens_summary = (sens.groupby(['brand_final','size_oz_rounded','STORE'], observed=True)
                    .agg(price_min=('opt_price','min'),
                         price_med=('opt_price','median'),
                         price_max=('opt_price','max'),
                         lift_min=('profit_lift_abs','min'),
                         lift_med=('profit_lift_abs','median'),
                         lift_max=('profit_lift_abs','max'))
                    .reset_index().round(3))

log('## 9. Sensitivity')
log(f'- grid: {len(beta_own_grid)} × {len(beta_cross_grid)} × {len(promo_grid)} '
    f'= {len(beta_own_grid)*len(beta_cross_grid)*len(promo_grid)} param combos')
log(f'- saved: {sens_path.relative_to(PROJECT_ROOT)}')
log('')
log('Top-10 sensitivity range (per cell):')
log('| brand | size | store | price [min, med, max] | lift [min, med, max] |')
log('|---|---|---|---|---|')
for _, r in sens_summary.iterrows():
    log(f'| {r["brand_final"]} | {r["size_oz_rounded"]:.2f} | {int(r["STORE"])} | '
        f'[{r["price_min"]:.2f}, {r["price_med"]:.2f}, {r["price_max"]:.2f}] | '
        f'[${r["lift_min"]:.1f}, ${r["lift_med"]:.1f}, ${r["lift_max"]:.1f}] |')


## 10. Save artifacts + figures + limitations


In [ ]:
# Save cell baselines (Streamlit will read this)
cell_path = PROCESSED / 'cell_baselines.parquet'
ce_opt.to_parquet(cell_path, index=False)

# Figure 1: portfolio top-10 lift bar
fig, ax = plt.subplots(figsize=(9, 4.5))
labels = [f"{r['brand_final'][:10]}|{r['size_oz_rounded']:.0f}oz|S{int(r['STORE'])}"
          for _, r in top10_diverse.iterrows()]
ax.barh(labels, top10_diverse['profit_lift_abs'].values, color='steelblue')
ax.set_xlabel('Predicted profit lift ($/week)')
ax.set_title('Top-10 portfolio recommendations — predicted weekly profit lift')
ax.invert_yaxis()
fig.tight_layout()
fig1_path = FIGURES / 'counterfactual_top10_lift.png'
fig.savefig(fig1_path, dpi=130)
plt.close(fig)

# Figure 2: sensitivity heatmap of recommended price for portfolio top-1 cell
top1 = top10_diverse.iloc[0]
sub = sens[(sens['brand_final']==top1['brand_final']) &
           (sens['size_oz_rounded']==top1['size_oz_rounded']) &
           (sens['STORE']==int(top1['STORE']))]
heat = sub.pivot_table(index='beta_own', columns='beta_cross', values='opt_price', aggfunc='mean')
fig, ax = plt.subplots(figsize=(5.5, 4))
im = ax.imshow(heat.values, aspect='auto', cmap='YlGnBu')
ax.set_xticks(range(heat.shape[1])); ax.set_xticklabels([f'{c:.2f}' for c in heat.columns])
ax.set_yticks(range(heat.shape[0])); ax.set_yticklabels([f'{r:.2f}' for r in heat.index])
ax.set_xlabel('β_cross'); ax.set_ylabel('β_own')
ax.set_title(f"Optimal price sensitivity\n{top1['brand_final']} {top1['size_oz_rounded']:.0f}oz Store {int(top1['STORE'])}")
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        ax.text(j, i, f"{heat.values[i,j]:.2f}", ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=ax, label='Recommended price ($)')
fig.tight_layout()
fig2_path = FIGURES / 'counterfactual_sensitivity_top1.png'
fig.savefig(fig2_path, dpi=130)
plt.close(fig)

log('## 10. Artifacts saved')
log(f'- {cell_path.relative_to(PROJECT_ROOT)}  (eligible cells + recommendations, parquet)')
log(f'- {(PROCESSED / "all_recommendations.csv").relative_to(PROJECT_ROOT)}')
log(f'- {(PROCESSED / "top_recommendations.csv").relative_to(PROJECT_ROOT)}')
log(f'- {(PROCESSED / "top_recommendations_diverse.csv").relative_to(PROJECT_ROOT)}')
log(f'- {(PROCESSED / "sensitivity_grid.csv").relative_to(PROJECT_ROOT)}')
log(f'- {fig1_path.relative_to(PROJECT_ROOT)}')
log(f'- {fig2_path.relative_to(PROJECT_ROOT)}')

log('')
log('## Limitations (to disclose in memo)')
log('1. **Cell-anchored prediction assumes log-linearity holds locally** —— extrapolations beyond '
    '`[0.85·p_min, 1.15·p_max]` are clipped by the grid; recommendations pegging the upper bound '
    '(flagged ⚠ in §8) should be treated as "raise, but verify with a smaller test" rather than '
    'a literal price target.')
log('2. **No retransformation correction needed for anchor predictions** —— `mean_q` is empirical '
    'level, not log; smearing factor S=1.14 from 03 applies to FE-based holdout prediction only.')
log('3. **Competitor price held fixed in main table** —— ignores game-theoretic response. '
    'Sensitivity grid varies β_cross × p_comp ±10% as a what-if.')
log('4. **Cross-cannibalization within own brand-size** not modeled —— raising 18oz price may shift '
    'demand to the 12oz SKU; current optimizer treats each cell independently.')
log('5. **Inventory not constrained** —— large predicted Q lifts assume restock capacity; flagged '
    'in top-10 view via `q_lift_ratio` column.')
log('6. **Margin uses AAC-derived unit_cost** —— see 03 limitation #2 (AAC ≠ marginal cost). '
    'Profit numbers are directionally useful, not exact accounting.')
log('7. **Eligibility filter (52-wk history, IQR price band) excludes new SKUs and clearance cells** '
    '—— intentional for MVP recommendation reliability; downstream can relax.')

# Write summary
out_path = REPORTS / 'counterfactual_summary.md'
out_path.write_text('\n'.join(summary_lines), encoding='utf-8')
log(f'\n- summary saved: {out_path.relative_to(PROJECT_ROOT)}')
